# T31-机构行为监测 - 可视化模块

## 1. 模块概述

可视化模块负责将分析结果转换为可视化图表数据格式，支持多种图表类型：

- **柱状图/条形图**: 机构净交易量、资产配置流向
- **热力图**: 机构间交易矩阵、城投利差地图
- **饼图/环形图**: 期限偏好、券种偏好
- **折线图**: 市场份额时序、行为指数时序
- **桑基图**: 资金流向
- **仪表盘/记分卡**: 机构行为评分

## 2. 图表数据格式转换器

In [ ]:
import pandas as pd
import numpy as np
from typing import Dict, List, Any, Optional
import json

class ChartDataFormatter:
    """图表数据格式转换器基类"""
    
    @staticmethod
    def to_bar_chart_data(df: pd.DataFrame, name_col: str = 'name', 
                         value_col: str = 'value') -> Dict[str, Any]:
        """
        转换为柱状图数据格式
        
        Args:
            df: 数据
            name_col: 名称列
            value_col: 值列
        
        Returns:
            Dict: 柱状图数据
        """
        return {
            'categories': df[name_col].tolist(),
            'values': df[value_col].tolist()
        }
    
    @staticmethod
    def to_pie_chart_data(df: pd.DataFrame, name_col: str = 'name',
                         value_col: str = 'value', 
                         percent_col: str = 'percent') -> List[Dict]:
        """
        转换为饼图数据格式
        
        Args:
            df: 数据
            name_col: 名称列
            value_col: 值列
            percent_col: 百分比列
        
        Returns:
            List[Dict]: 饼图数据列表
        """
        result = []
        for _, row in df.iterrows():
            item = {
                'name': row[name_col],
                'value': row[value_col]
            }
            if percent_col in df.columns:
                item['percent'] = row[percent_col]
            result.append(item)
        return result
    
    @staticmethod
    def to_line_chart_data(df: pd.DataFrame, date_col: str = 'date',
                          value_cols: List[str] = None) -> Dict[str, Any]:
        """
        转换为折线图数据格式
        
        Args:
            df: 数据
            date_col: 日期列
            value_cols: 值列列表
        
        Returns:
            Dict: 折线图数据
        """
        if value_cols is None:
            value_cols = [col for col in df.columns if col != date_col]
        
        return {
            'dates': df[date_col].astype(str).tolist(),
            'series': {col: df[col].tolist() for col in value_cols}
        }

# 创建格式转换器
formatter = ChartDataFormatter()
print("图表数据格式转换器已创建")

## 3. 图表1：各机构类型每日净交易量

In [ ]:
class Chart1Formatter:
    """图表1数据格式化器"""
    
    @staticmethod
    def format(df: pd.DataFrame) -> Dict[str, Any]:
        """
        格式化图表1数据
        
        Args:
            df: 包含name和value列的数据
        
        Returns:
            Dict: 图表数据
        """
        # 确保数据格式正确
        df = df.copy()
        
        # 按值排序
        df = df.sort_values('value', ascending=False)
        
        return {
            'chart_type': 'bar',
            'data': {
                'categories': df['name'].tolist(),
                'values': df['value'].round(2).tolist()
            },
            'options': {
                'title': '各机构类型每日净交易量',
                'xAxis': {'type': 'category', 'name': '机构类型'},
                'yAxis': {'type': 'value', 'name': '净买入量（亿元）'},
                'tooltip': {'trigger': 'axis'}
            }
        }

print("图表1格式化器已创建")

## 4. 图表2：机构间交易矩阵（热力图）

In [ ]:
class Chart2Formatter:
    """图表2数据格式化器 - 机构间交易矩阵"""
    
    @staticmethod
    def format(trade_matrix: pd.DataFrame, date_str: str) -> Dict[str, Any]:
        """
        格式化热力图数据
        
        Args:
            trade_matrix: 交易矩阵（行为卖方，列为买方）
            date_str: 日期字符串
        
        Returns:
            Dict: 热力图数据
        """
        seller_types = trade_matrix.index.tolist()
        buyer_types = trade_matrix.columns.tolist()
        
        heatmap_data = []
        for y, seller in enumerate(seller_types):
            for x, buyer in enumerate(buyer_types):
                amount = trade_matrix.loc[seller, buyer]
                heatmap_data.append({
                    'x': x,
                    'y': y,
                    'value': round(amount, 2),
                    'value2': round(amount, 2),
                    'value3': round(amount, 2)
                })
        
        return {
            'chart_type': 'heatmap',
            'data': {
                'heatmap_data': heatmap_data,
                'categories_table': [
                    {'type': 'cat', 'v': buyer_types},
                    {'type': 'cat2', 'v': seller_types}
                ],
                'text': f'数据截止 {date_str}'
            },
            'options': {
                'title': '机构间交易矩阵',
                'xAxis': {'type': 'category', 'data': buyer_types, 'name': '买方'},
                'yAxis': {'type': 'category', 'data': seller_types, 'name': '卖方'},
                'visualMap': {'min': 0, 'max': trade_matrix.values.max()}
            }
        }

print("图表2格式化器已创建")

## 5. 图表3/4：机构期限/券种偏好分析（饼图）

In [ ]:
class Chart3_4Formatter:
    """图表3/4数据格式化器 - 期限/券种偏好"""
    
    @staticmethod
    def format(df: pd.DataFrame, title: str, institution_type: str) -> Dict[str, Any]:
        """
        格式化饼图数据
        
        Args:
            df: 包含name, value, percent列的数据
            title: 图表标题
            institution_type: 机构类型
        
        Returns:
            Dict: 饼图数据
        """
        # 确保数据有效
        df = df[df['value'] > 0].copy()
        
        pie_data = []
        for _, row in df.iterrows():
            pie_data.append({
                'name': row['name'],
                'value': round(row['value'], 2),
                'percent': round(row.get('percent', 0), 2)
            })
        
        return {
            'chart_type': 'pie',
            'data': pie_data,
            'options': {
                'title': f'{institution_type} - {title}',
                'tooltip': {
                    'trigger': 'item',
                    'formatter': '{b}: {c} ({d}%)'
                },
                'legend': {'orient': 'vertical', 'left': 'left'}
            }
        }

print("图表3/4格式化器已创建")

## 6. 图表5：市场份额时序分析（折线图）

In [ ]:
class Chart5Formatter:
    """图表5数据格式化器 - 市场份额时序"""
    
    @staticmethod
    def format(df: pd.DataFrame) -> Dict[str, Any]:
        """
        格式化折线图数据
        
        Args:
            df: 透视图格式的市场份额数据（日期为行，机构类型为列）
        
        Returns:
            Dict: 折线图数据
        """
        # 转换日期为字符串
        dates = df.index.astype(str).tolist()
        
        series = []
        for col in df.columns:
            series.append({
                'name': col,
                'data': df[col].round(2).tolist()
            })
        
        return {
            'chart_type': 'line',
            'data': {
                'dates': dates,
                'series': series
            },
            'options': {
                'title': '市场份额时序分析',
                'xAxis': {'type': 'category', 'data': dates},
                'yAxis': {'type': 'value', 'name': '市场份额(%)'},
                'legend': {'data': df.columns.tolist()},
                'tooltip': {'trigger': 'axis'}
            }
        }

print("图表5格式化器已创建")

## 7. 图表7：城投利差热力图

In [ ]:
class Chart7Formatter:
    """图表7数据格式化器 - 城投利差热力图"""
    
    @staticmethod
    def format(df: pd.DataFrame) -> Dict[str, Any]:
        """
        格式化地图热力图数据
        
        Args:
            df: 包含name(省份)和value(利差)的数据
        
        Returns:
            Dict: 地图热力图数据
        """
        map_data = []
        for _, row in df.iterrows():
            map_data.append({
                'name': row['name'],
                'value': round(row['value'], 2)
            })
        
        # 计算颜色范围
        min_val = df['value'].min()
        max_val = df['value'].max()
        
        return {
            'chart_type': 'map',
            'data': map_data,
            'options': {
                'title': '全国城投利差热力图',
                'visualMap': {
                    'min': round(min_val, 2),
                    'max': round(max_val, 2),
                    'text': ['高', '低'],
                    'calculable': True
                },
                'tooltip': {
                    'trigger': 'item',
                    'formatter': '{b}: {c} BP'
                }
            }
        }

print("图表7格式化器已创建")

## 8. 图表8：市场核心行为指数

In [ ]:
class Chart8Formatter:
    """图表8数据格式化器 - 市场核心行为指数"""
    
    @staticmethod
    def format(df: pd.DataFrame) -> Dict[str, Any]:
        """
        格式化多系列折线图数据
        
        Args:
            df: 包含date, rfi_premium, herding, duration_net_flow的数据
        
        Returns:
            Dict: 多系列折线图数据
        """
        dates = df['date'].astype(str).tolist()
        
        series = [
            {
                'name': 'RFI Premium',
                'data': df['rfi_premium'].round(4).tolist()
            },
            {
                'name': 'Herding Index',
                'data': df.get('herding', pd.Series([0]*len(df))).round(4).tolist()
            },
            {
                'name': 'Duration Net Flow',
                'data': df['duration_net_flow'].round(2).tolist()
            }
        ]
        
        return {
            'chart_type': 'multi_line',
            'data': {
                'dates': dates,
                'series': series
            },
            'options': {
                'title': '市场核心行为指数',
                'xAxis': {'type': 'category', 'data': dates},
                'yAxis': [
                    {'type': 'value', 'name': 'RFI/Herding'},
                    {'type': 'value', 'name': 'Duration'}
                ],
                'legend': {'data': ['RFI Premium', 'Herding Index', 'Duration Net Flow']},
                'tooltip': {'trigger': 'axis'}
            }
        }

print("图表8格式化器已创建")

## 9. 图表9：市场风险定价

In [ ]:
class Chart9Formatter:
    """图表9数据格式化器 - 市场风险定价"""
    
    @staticmethod
    def format(df: pd.DataFrame) -> Dict[str, Any]:
        """
        格式化双轴折线图数据
        
        Args:
            df: 包含date, term_spread, credit_spread的数据
        
        Returns:
            Dict: 双轴折线图数据
        """
        dates = df['date'].astype(str).tolist()
        
        series = [
            {
                'name': '期限利差(10Y-1Y)',
                'data': df['term_spread'].round(2).tolist(),
                'yAxisIndex': 0
            },
            {
                'name': '信用利差(AA+ 3Y)',
                'data': df.get('credit_spread', pd.Series([0]*len(df))).round(2).tolist(),
                'yAxisIndex': 1
            }
        ]
        
        return {
            'chart_type': 'dual_line',
            'data': {
                'dates': dates,
                'series': series
            },
            'options': {
                'title': '市场风险定价',
                'xAxis': {'type': 'category', 'data': dates},
                'yAxis': [
                    {'type': 'value', 'name': '期限利差(%)'},
                    {'type': 'value', 'name': '信用利差(BP)'}
                ],
                'legend': {'data': ['期限利差(10Y-1Y)', '信用利差(AA+ 3Y)']},
                'tooltip': {'trigger': 'axis'}
            }
        }

print("图表9格式化器已创建")

## 10. 图表10/11：记分卡和温度计

In [ ]:
class Chart10_11Formatter:
    """图表10/11数据格式化器 - 记分卡和温度计"""
    
    @staticmethod
    def format_scorecard(data: Dict[str, Any], institution_type: str) -> Dict[str, Any]:
        """
        格式化记分卡数据
        
        Args:
            data: 评分数据字典
            institution_type: 机构类型
        
        Returns:
            Dict: 记分卡数据
        """
        return {
            'chart_type': 'scorecard',
            'data': {
                'title': f'{institution_type} 行为记分卡',
                'indicators': [
                    {
                        'name': 'RFY Premium',
                        'value': round(data.get('rfi_score', 0), 4),
                        'change': round(data.get('rfi_change', 0), 4)
                    },
                    {
                        'name': 'Herding Index',
                        'value': round(data.get('herding_score', 0), 4),
                        'change': round(data.get('herding_change', 0), 4)
                    },
                    {
                        'name': 'Duration Flow',
                        'value': round(data.get('duration_flow_score', 0), 2),
                        'change': round(data.get('duration_flow_change', 0), 2)
                    }
                ]
            }
        }
    
    @staticmethod
    def format_thermometer(data: Dict[str, Any]) -> Dict[str, Any]:
        """
        格式化温度计数据
        
        Args:
            data: 资金面数据
        
        Returns:
            Dict: 温度计数据
        """
        return {
            'chart_type': 'thermometer',
            'data': {
                'title': '资金面与杠杆温度计',
                'indicators': [
                    {
                        'name': '回购余额',
                        'value': round(data.get('total_repo_balance', 0), 2),
                        'unit': '亿元'
                    },
                    {
                        'name': 'DR007',
                        'value': round(data.get('dr007', 0), 4),
                        'unit': '%'
                    },
                    {
                        'name': 'R007',
                        'value': round(data.get('r007', 0), 4),
                        'unit': '%'
                    }
                ],
                'date': str(data.get('date', ''))
            }
        }

print("图表10/11格式化器已创建")

## 11. 图表12：桑基图

In [ ]:
class Chart12Formatter:
    """图表12数据格式化器 - 资产配置流向桑基图"""
    
    @staticmethod
    def format_sankey(df: pd.DataFrame, institution_type: str) -> Dict[str, Any]:
        """
        格式化桑基图数据
        
        Args:
            df: 包含source, target, value的数据
            institution_type: 机构类型
        
        Returns:
            Dict: 桑基图数据
        """
        # 构建节点列表
        nodes_set = set()
        for _, row in df.iterrows():
            nodes_set.add(row['source'])
            nodes_set.add(row['target'])
        
        nodes = [{'name': name} for name in list(nodes_set)]
        
        # 构建链接列表
        links = []
        for _, row in df.iterrows():
            if row['value'] > 0:
                links.append({
                    'source': row['source'],
                    'target': row['target'],
                    'value': round(row['value'], 2)
                })
        
        return {
            'chart_type': 'sankey',
            'data': {
                'nodes': nodes,
                'links': links
            },
            'options': {
                'title': f'{institution_type} 大类券种配置流向',
                'tooltip': {'trigger': 'item'}
            }
        }
    
    @staticmethod
    def format_waterfall(df: pd.DataFrame, institution_type: str) -> Dict[str, Any]:
        """
        格式化瀑布图数据
        
        Args:
            df: 包含name和value的数据
            institution_type: 机构类型
        
        Returns:
            Dict: 瀑布图数据
        """
        # 添加合计行
        total = df['value'].sum()
        df_with_total = pd.concat([
            df,
            pd.DataFrame([{'name': '总净流入', 'value': total}])
        ], ignore_index=True)
        
        return {
            'chart_type': 'waterfall',
            'data': {
                'categories': df_with_total['name'].tolist(),
                'values': df_with_total['value'].round(2).tolist()
            },
            'options': {
                'title': f'{institution_type} 大类券种配置流向',
                'xAxis': {'type': 'category'},
                'yAxis': {'type': 'value', 'name': '净流入（亿元）'}
            }
        }

print("图表12格式化器已创建")

## 12. 统一图表生成器

In [ ]:
class ChartGenerator:
    """统一图表生成器"""
    
    def __init__(self):
        self.chart1 = Chart1Formatter()
        self.chart2 = Chart2Formatter()
        self.chart3_4 = Chart3_4Formatter()
        self.chart5 = Chart5Formatter()
        self.chart7 = Chart7Formatter()
        self.chart8 = Chart8Formatter()
        self.chart9 = Chart9Formatter()
        self.chart10_11 = Chart10_11Formatter()
        self.chart12 = Chart12Formatter()
    
    def generate_chart(self, chart_id: int, df: pd.DataFrame, **kwargs) -> Dict[str, Any]:
        """
        生成指定图表
        
        Args:
            chart_id: 图表编号(1-14)
            df: 数据
            **kwargs: 其他参数
        
        Returns:
            Dict: 图表数据
        """
        if chart_id == 1:
            return self.chart1.format(df)
        elif chart_id == 2:
            return self.chart2.format(df, kwargs.get('date_str', ''))
        elif chart_id in [3, 4]:
            return self.chart3_4.format(
                df, 
                kwargs.get('title', ''),
                kwargs.get('institution_type', '')
            )
        elif chart_id == 5:
            return self.chart5.format(df)
        elif chart_id == 7:
            return self.chart7.format(df)
        elif chart_id == 8:
            return self.chart8.format(df)
        elif chart_id == 9:
            return self.chart9.format(df)
        elif chart_id in [10, 11]:
            if kwargs.get('chart_type') == 'thermometer':
                return self.chart10_11.format_thermometer(df if isinstance(df, dict) else df.to_dict())
            else:
                return self.chart10_11.format_scorecard(
                    df if isinstance(df, dict) else df.to_dict(),
                    kwargs.get('institution_type', '')
                )
        elif chart_id == 12:
            if kwargs.get('chart_type') == 'sankey':
                return self.chart12.format_sankey(df, kwargs.get('institution_type', ''))
            else:
                return self.chart12.format_waterfall(df, kwargs.get('institution_type', ''))
        else:
            return {'error': f'图表{chart_id}格式化器未实现'}

# 创建图表生成器
chart_generator = ChartGenerator()
print("统一图表生成器已创建")

## 13. 模块总结

In [ ]:
# 模块总结
print("=" * 60)
print("可视化模块组件列表")
print("=" * 60)
print("\n1. ChartDataFormatter - 基础数据格式转换器")
print("   - to_bar_chart_data(): 柱状图格式")
print("   - to_pie_chart_data(): 饼图格式")
print("   - to_line_chart_data(): 折线图格式")
print("\n2. Chart1Formatter - 图表1：各机构类型每日净交易量")
print("3. Chart2Formatter - 图表2：机构间交易矩阵（热力图）")
print("4. Chart3_4Formatter - 图表3/4：期限/券种偏好（饼图）")
print("5. Chart5Formatter - 图表5：市场份额时序（折线图）")
print("6. Chart7Formatter - 图表7：城投利差热力图（地图）")
print("7. Chart8Formatter - 图表8：市场核心行为指数（多系列折线）")
print("8. Chart9Formatter - 图表9：市场风险定价（双轴折线）")
print("9. Chart10_11Formatter - 图表10/11：记分卡和温度计")
print("10. Chart12Formatter - 图表12：资产配置流向（桑基图/瀑布图）")
print("\n11. ChartGenerator - 统一图表生成器")
print("    - generate_chart(): 根据图表ID生成图表数据")
print("\n" + "=" * 60)
print("可视化模块加载完成")
print("=" * 60)